In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:100% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:20pt;}
div.text_cell_render.rendered_html{font-size:18pt;}
div.text_cell_render.rendered_html{font-size:15pt;}
div.output {font-size:18pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:18pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:18pt;padding:5px;}
table.dataframe{font-size:18px;}
</style>
"""))

[ RAG 구현 절차 ]

```
1.	문서의 내용을 읽는다(document_loader를 이용)
(1)	https://python.langchain.com/v0.2/docs/integrations/document_loaders/ 
(2)	https://python.langchain.com/v0.2/docs/integrations/document_loaders/microsoft_word/
%pip install --upgrade --quiet  docx2txt
2.	문서를 쪼갠다(한번에 이해하고 처리할 수 있는 입력+출력 토큰수가 제한)
(1)	 https://python.langchain.com/v0.2/docs/how_to/recursive_text_splitter/#splitting-text-from-languages-without-word-boundaries 
%pip install -qU langchain-text-splitters
3.	쪼갠 문서를 임베딩하여 vector database에 넣음
(1)	OpenAIEmbeddings나 UpstageEmbeddings이용해서 임베딩
(2)	https://python.langchain.com/v0.2/docs/integrations/vectorstores/chroma/  
%pip install –q langchain-chroma
4.	질문을 이용해 유사도 검색
5.	유사도 검색한 문서를 LLM에 질문으로 전달하여 답변 얻음(제공되는 Prompt활용)
(1)	https://python.langchain.com/v0.2/docs/tutorials/rag/
%pip install –q langchain langchainhub
http://smith.langchain.com에서 key생성 .env key(LANGCHAIN_API_KEY) 추가

```

# 2. 문서를 쪼개면서 읽기(o)

In [1]:
import time
start = time.time()
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader('./tax_docs/소득세법(법률)(제20615호)(20250701).docx')
text_splitter = RecursiveCharacterTextSplitter(  # 문서를 쪼개는 기준이 문자수
    chunk_size=1500, #문서를 쪼갤때 1500글자씩 쪼개
    chunk_overlap=200
)
# 1번째 chunk 1~1450글자
# 2번째 chunk 1250~1750글자
documents = loader.load_and_split(text_splitter=text_splitter)
runtime = time.time() - start
print('문서 쪼개면서 읽는 시간 :', runtime)

문서 쪼개면서 읽는 시간 : 5.265929698944092


# 3. 쪼갠문서를 임베딩 -> 벡터 데이터베이스 저장
- 임베딩 모델 : upstage의 text-embedding-3-large (기본:text-embedding-ada-002)
- 벡터 데이터베이스 : chroma

In [2]:
# https://python.langchain.com/v0.2/docs/integrations/text_embedding/upstage
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
load_dotenv()
embeddings = UpstageEmbeddings(
    model="solar-embedding-1-large"
    # model="embedding-query"
)

In [4]:
doc_result = embeddings.embed_documents(
    ["소득세법 어쩌구 저쩌구", documents[0].page_content]
)
print(len(doc_result), len(doc_result[0]), len(doc_result[1]))

2 4096 4096


In [15]:
len(embeddings), len(embeddings[0]), len(embeddings[1])

(2, 3072, 3072)

In [16]:
len(embedding.embed_query("소득세"))

3072

In [11]:
from langchain_chroma import Chroma
# 데이터를 처음 저장할 때
# database = Chroma.from_documents(                                 
#     documents=documents,
#     embedding=embeddings,
#     collection_name="tax-collection", # 생략시 이름 랜덤
#     persist_directory='chroma_upstage'      # 생략시 로컬데이터베이스에 저장안됨. 프로그램 종료시 db날라감
# )
# 이미 저장된 vector DB를 사용할 때
database = Chroma(
    embedding_function=embeddings,
    collection_name="tax-collection",
    persist_directory='chroma_upstage'
)

# 4. vector DB에 질문과 유사도 검색(답변 생성을 위한 retrieval)

In [12]:
query = "연봉 5천만원인 직장인의 소득세는 얼마인가요?"
retrieved_docs = database.similarity_search(query,
                                           k=3) # 기본 k는 4

In [13]:
retrieved_docs

[Document(id='8f0911ac-7bae-472a-a845-906dc27b5a5b', metadata={'source': './tax_docs/소득세법(법률)(제20615호)(20250701).docx'}, page_content='2. 2명인 경우: 연 55만원\n\n3. 3명 이상인 경우: 연 55만원과 2명을 초과하는 1명당 연 40만원을 합한 금액\n\n② 삭제<2017. 12. 19.>\n\n③ 해당 과세기간에 출산하거나 입양 신고한 공제대상자녀가 있는 경우 다음 각 호의 구분에 따른 금액을 종합소득산출세액에서 공제한다.<신설 2015. 5. 13., 2016. 12. 20.>\n\n1. 출산하거나 입양 신고한 공제대상자녀가 첫째인 경우: 연 30만원\n\n2. 출산하거나 입양 신고한 공제대상자녀가 둘째인 경우: 연 50만원\n\n3. 출산하거나 입양 신고한 공제대상자녀가 셋째 이상인 경우: 연 70만원\n\n④ 제1항 및 제3항에 따른 공제를 “자녀세액공제”라 한다.<신설 2015. 5. 13., 2017. 12. 19.>\n\n[본조신설 2014. 1. 1.]\n\n[종전 제59조의2는 제59조의5로 이동 <2014. 1. 1.>]\n\n\n\n제59조의3(연금계좌세액공제) ① 종합소득이 있는 거주자가 연금계좌에 납입한 금액 중 다음 각 호에 해당하는 금액을 제외한 금액(이하 “연금계좌 납입액”이라 한다)의 100분의 12[해당 과세기간에 종합소득과세표준을 계산할 때 합산하는 종합소득금액이 4천 500만원 이하(근로소득만 있는 경우에는 총급여액 5천 500만원 이하)인 거주자에 대해서는 100분의 15]에 해당하는 금액을 해당 과세기간의 종합소득산출세액에서 공제한다. 다만, 연금계좌 중 연금저축계좌에 납입한 금액이 연 600만원을 초과하는 경우에는 그 초과하는 금액은 없는 것으로 하고, 연금저축계좌에 납입한 금액 중 600만원 이내의 금액과 퇴직연금계좌에 납입한 금액을 합한 금액이 연 900만원을 초과하는 경우에는 그 초과하는 금액은 없는

# 5. 유사도 검색으로 가져온 문서를 질문과 같이 LLM 전달하여 답변 생성

In [15]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4.1-nano")

In [16]:
prompt = f"""[identity]
- 당신은 최고의 한국 소득세 전문자입니다
- [context]를 참고해서 사용자의 질문에 답변해 주세요
[context]는 다음과 같아요
{retrieved_docs}
Question : {query}"""

In [ ]:
ai_message = qa_chain.invoke(prompt)

In [18]:
print(ai_message.content)

연봉이 5천만원인 직장인의 소득세를 계산하기 위해서는 여러 공제와 세율을 고려해야 합니다. 아래는 기본적인 계산 방법을 안내드립니다.

1. **근로소득공제**  
- 근로소득공제는 총급여액에 따라 차등 적용되며, 5천만원의 경우 2천만원 한도로 공제됩니다.  
- 따라서, 근로소득공제액 = 2,000만원입니다.

2. **과세표준 산출**  
- 과세표준 = 총급여액 – 근로소득공제  
- 과세표준 = 50,000,000 – 20,000,000 = 30,000,000원

3. **기본 세율 적용**  
- 2023년 기준 소득세 세율표에 따르면,  
  - 1,200만원 이하: 6%  
  - 1,200만원 초과 4,600만원 이하: 15% = 1,200만원 × 6% + (과세표준 – 1,200만원) × 15%  

하지만 구체적인 세액 계산을 위해서 표를 활용하는 것이 가장 좋습니다.

4. **가정하여 계산**  
- 과세표준 3,000만원에 대해,  
  - 첫 1,200만원: 1,200만원 × 6% = 72만원  
  - 나머지 1,800만원: 1,800만원 × 15% = 270만원  
- 세액 합계: 72만원 + 270만원 = 342만원

5. **세액공제 고려**  
- 기본공제, 자녀공제, 보험료공제, 연금계좌공제 등 각종 공제에 따라 최종 세액이 결정됩니다.  
- 만약 별다른 공제 사항이 없다면, 대략 342만원이 부과됩니다.

---

**요약:**  
연봉 5천만원인 직장인은 기본적인 세액 계산 시 약 **342만원**의 소득세 부담이 예상됩니다. 그러나 개인별 공제 항목이나 특별 공제, 세액 감면 여부에 따라 최종 세액은 달라질 수 있습니다. 정확한 계산은 종합소득세 신고 시 세무 전문가와 상담하시는 것이 좋습니다.


# 5. Augmentation을 위한 제공되는 Prompt활용하여 langchain으로 답변 생성

In [19]:
query = "연봉 5천만원인 직장인의 소득세는 얼마인가요?"

from langchain import hub
prompt = hub.pull("rlm/rag-prompt")
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'rlm', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '50442af133e61576e74536c6556cefe1fac147cad032f4377b60c436e6cdcb6e'}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})])

### RetrievalQA를 통해 LLM전달 (create_retrieval_chain이 대체)
```
 query -> retriever전달(백터 검색 수행) 
 -> retrieval문서 -> prompt의 {context}에 삽입
 -> query -> prompt의 {question}에 삽입
```

In [20]:
from langchain.chains import RetrievalQA
qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever = database.as_retriever(search_kwargs={'k':5}),
    chain_type_kwargs={"prompt":prompt}
)

In [21]:
ai_message = qa_chain.invoke({"query":query})

In [22]:
ai_message

{'query': '연봉 5천만원인 직장인의 소득세는 얼마인가요?',
 'result': '연봉 5천만원인 직장인의 소득세는 공제 후 종합소득세 계산에 따라 달라지며, 예를 들어 근로소득공제(최대 2000만원)와 자녀세액공제, 연금계좌공제 등을 고려해야 합니다. 일반적으로 총급여액이 5천만원 이하인 경우, 근로소득공제는 최대 2000만원까지 적용되며, 세율은 점차 증가하는 구간별 세율이 적용됩니다. 따라서 정확한 세액은 공제와 세율 구간에 따라 다르며, 구체적인 계산이 필요합니다.'}